# Upstox-Integrated Trade-Position Monitor + IV–RV Context Dashboard

## Mission
Create a comprehensive trading dashboard that provides:
1. **IV-RV Context Dashboard**: Real-time implied volatility analysis with IVR/IVP, realized volatility, VRP, and regime classification
2. **Trade-Position Monitor**: Live portfolio P&L tracking with Greeks analysis and stress testing

## Features
- Live Upstox v3 API integration (read-only)
- Real-time market data via WebSocket
- Advanced volatility analytics (IVR, IVP, VRP, term structure)
- Portfolio Greeks monitoring and stress testing
- Action prompts for delta hedging and position management
- India VIX regime overlay

## Scale
- ~60-90 liquid single-stock underlyings (NSE only)
- No order placement - read-only decision support

---

## 1. CONFIG - API Credentials

**Instructions**: Paste your Upstox API credentials in the cell below. Create a `.env` file in the same directory with:
```
UPSTOX_API_KEY=your_api_key
UPSTOX_API_SECRET=your_api_secret
UPSTOX_REDIRECT_URI=your_redirect_uri
UPSTOX_ACCESS_TOKEN=your_access_token
```

In [ ]:
# CONFIG SECTION - Paste your Upstox API credentials here
import os
from dotenv import load_dotenv
from pydantic import BaseSettings
from typing import Optional

# Load environment variables
load_dotenv()

class UpstoxConfig(BaseSettings):
    api_key: str
    api_secret: str
    redirect_uri: str
    access_token: Optional[str] = None
    
    class Config:
        env_file = ".env"

# Initialize config
config = UpstoxConfig(
    api_key=os.getenv('UPSTOX_API_KEY', 'YOUR_API_KEY_HERE'),
    api_secret=os.getenv('UPSTOX_API_SECRET', 'YOUR_API_SECRET_HERE'),
    redirect_uri=os.getenv('UPSTOX_REDIRECT_URI', 'YOUR_REDIRECT_URI_HERE'),
    access_token=os.getenv('UPSTOX_ACCESS_TOKEN', 'YOUR_ACCESS_TOKEN_HERE')
)

print(f"API Key configured: {config.api_key[:8]}...")
print(f"Access Token configured: {bool(config.access_token)}")

## 2. Imports and Dependencies

In [ ]:
# Core libraries
import asyncio
import json
import time
import warnings
from datetime import datetime, timedelta
from typing import Dict, List, Optional, Tuple, Any
from dataclasses import dataclass
from pathlib import Path

# Data processing
import pandas as pd
import numpy as np
from scipy import stats
import statsmodels.api as sm
from statsmodels.tsa.holtwinters import ExponentialSmoothing

# HTTP and async
import httpx
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type

# Visualization
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output

# Logging
from loguru import logger

# Suppress warnings
warnings.filterwarnings('ignore')

# Configure logging
logger.remove()
logger.add(lambda msg: print(msg, end=""), level="INFO")
logger.add("upstox_monitor.log", level="DEBUG", rotation="1 day")

print("✅ All dependencies imported successfully")
print(f"📅 Current time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S IST')}")

## 3. Helpers - Authentication and API Client

In [ ]:
# Data classes for structured data
@dataclass
class OptionData:
    symbol: str
    strike: float
    expiry: str
    option_type: str
    ltp: float
    iv: float
    delta: float
    gamma: float
    theta: float
    vega: float
    underlying: str
    dte: int

@dataclass
class PositionData:
    symbol: str
    quantity: int
    average_price: float
    ltp: float
    pnl: float
    pnl_percentage: float
    day_pnl: float
    day_pnl_percentage: float

@dataclass
class VolatilityMetrics:
    symbol: str
    current_iv: float
    iv_rank: float
    iv_percentile: float
    realized_vol: float
    vrp: float
    term_slope: float
    skew: float
    atr: float
    expected_move: float
    vix_regime: str
    bias: str  # GREEN, SKY-BLUE, RED

print("✅ Data classes defined")

In [ ]:
# Upstox API Client with retry logic
class UpstoxAPIClient:
    def __init__(self, config):
        self.config = config
        self.base_url = "https://api.upstox.com/v3"
        self.ws_url = None
        self.client = None
        self.session = None
        
    async def __aenter__(self):
        self.client = httpx.AsyncClient(timeout=30.0)
        return self
        
    async def __aexit__(self, exc_type, exc_val, exc_tb):
        if self.client:
            await self.client.aclose()
    
    def _get_headers(self):
        return {
            "Authorization": f"Bearer {self.config.access_token}",
            "Content-Type": "application/json",
            "Accept": "application/json"
        }
    
    @retry(
        stop=stop_after_attempt(3),
        wait=wait_exponential(multiplier=1, min=4, max=10),
        retry=retry_if_exception_type((httpx.HTTPStatusError, httpx.ConnectError))
    )
    async def _make_request(self, method: str, endpoint: str, **kwargs):
        url = f"{self.base_url}{endpoint}"
        headers = self._get_headers()
        
        try:
            response = await self.client.request(
                method, url, headers=headers, **kwargs
            )
            response.raise_for_status()
            return response.json()
        except httpx.HTTPStatusError as e:
            if e.response.status_code == 429:
                logger.warning(f"Rate limit hit for {endpoint}, retrying...")
                raise
            else:
                logger.error(f"HTTP error {e.response.status_code} for {endpoint}: {e.response.text}")
                raise
        except Exception as e:
            logger.error(f"Request failed for {endpoint}: {str(e)}")
            raise
    
    async def get_instruments(self, exchange: str = "NSE_FO"):
        """Get instruments for the specified exchange"""
        return await self._make_request("GET", f"/instruments/{exchange}")
    
    async def get_greeks(self, instrument_keys: List[str]):
        """Get option Greeks for up to 50 instruments"""
        if len(instrument_keys) > 50:
            raise ValueError("Maximum 50 instrument keys allowed per request")
        
        payload = {"instrument_keys": instrument_keys}
        return await self._make_request("POST", "/option-greeks", json=payload)
    
    async def get_positions(self):
        """Get current positions"""
        return await self._make_request("GET", "/portfolio/positions")
    
    async def get_ohlc(self, instrument_key: str, interval: str = "1m", from_date: str = None, to_date: str = None):
        """Get OHLC data for an instrument"""
        params = {"interval": interval}
        if from_date:
            params["from_date"] = from_date
        if to_date:
            params["to_date"] = to_date
            
        return await self._make_request("GET", f"/historical/ohlc/{instrument_key}", params=params)
    
    async def get_authorize_url(self):
        """Get WebSocket authorization URL"""
        return await self._make_request("GET", "/market-data/feed/authorize")

print("✅ Upstox API Client defined")

In [ ]:
# Utility functions
def is_trading_hours() -> bool:
    """Check if current time is within NSE trading hours (IST)"""
    now = datetime.now()
    # NSE trading hours: 9:15 AM to 3:30 PM IST
    trading_start = now.replace(hour=9, minute=15, second=0, microsecond=0)
    trading_end = now.replace(hour=15, minute=30, second=0, microsecond=0)
    
    # Check if it's a weekday (Monday=0, Sunday=6)
    if now.weekday() >= 5:  # Saturday or Sunday
        return False
    
    return trading_start <= now <= trading_end

def calculate_dte(expiry_date: str) -> int:
    """Calculate days to expiry"""
    try:
        expiry = datetime.strptime(expiry_date, "%Y-%m-%d")
        today = datetime.now().date()
        return (expiry.date() - today).days
    except:
        return 0

def batch_instruments(instrument_keys: List[str], batch_size: int = 50) -> List[List[str]]:
    """Batch instrument keys for API requests"""
    return [instrument_keys[i:i + batch_size] for i in range(0, len(instrument_keys), batch_size)]

def calculate_iv_rank(current_iv: float, iv_history: List[float]) -> Tuple[float, float]:
    """Calculate IV Rank and IV Percentile"""
    if not iv_history or len(iv_history) < 2:
        return 0.0, 0.0
    
    iv_min = min(iv_history)
    iv_max = max(iv_history)
    
    if iv_max == iv_min:
        return 0.5, 0.5
    
    iv_rank = (current_iv - iv_min) / (iv_max - iv_min)
    iv_percentile = sum(1 for iv in iv_history if iv < current_iv) / len(iv_history)
    
    return iv_rank, iv_percentile

def calculate_realized_volatility(returns: pd.Series, window: int = 20) -> float:
    """Calculate realized volatility using EWMA"""
    if len(returns) < 2:
        return 0.0
    
    # Use EWMA with lambda=0.94
    lambda_param = 0.94
    weights = [(1 - lambda_param) * (lambda_param ** i) for i in range(len(returns))]
    weights = weights[::-1]  # Reverse to match chronological order
    
    # Calculate weighted variance
    weighted_var = np.average(returns ** 2, weights=weights)
    
    # Annualize (assuming daily returns)
    return np.sqrt(weighted_var * 252)

def calculate_atr(high: pd.Series, low: pd.Series, close: pd.Series, window: int = 14) -> float:
    """Calculate Average True Range"""
    if len(high) < window:
        return 0.0
    
    high_low = high - low
    high_close = np.abs(high - close.shift(1))
    low_close = np.abs(low - close.shift(1))
    
    true_range = np.maximum(high_low, np.maximum(high_close, low_close))
    atr = true_range.rolling(window=window).mean().iloc[-1]
    
    return atr if not pd.isna(atr) else 0.0

def calculate_expected_move(spot: float, iv: float, days: int) -> float:
    """Calculate expected move over N days"""
    return spot * iv * np.sqrt(days / 252)

print("✅ Utility functions defined")

## 4. Data Pipelines - Instruments, WebSocket, Greeks, Positions, OHLC

In [ ]:
# Universe Builder - Load and filter NSE F&O instruments
class UniverseBuilder:
    def __init__(self, api_client):
        self.api_client = api_client
        self.instruments_df = None
        self.option_universe = None
        self.cache_file = Path("instruments_cache.csv")
    
    async def load_instruments(self, force_refresh: bool = False):
        """Load NSE F&O instruments and filter for liquid single-stock options"""
        # Check cache first
        if not force_refresh and self.cache_file.exists():
            cache_time = datetime.fromtimestamp(self.cache_file.stat().st_mtime)
            if (datetime.now() - cache_time).hours < 6:  # Cache valid for 6 hours
                logger.info("Loading instruments from cache")
                self.instruments_df = pd.read_csv(self.cache_file)
                return self._filter_universe()
        
        # Load from API
        logger.info("Loading instruments from Upstox API")
        try:
            response = await self.api_client.get_instruments("NSE_FO")
            
            if response.get("status") == "success":
                instruments = response.get("data", [])
                self.instruments_df = pd.DataFrame(instruments)
                
                # Cache the data
                self.instruments_df.to_csv(self.cache_file, index=False)
                logger.info(f"Loaded {len(self.instruments_df)} instruments from API")
                
                return self._filter_universe()
            else:
                logger.error(f"Failed to load instruments: {response}")
                return None
                
        except Exception as e:
            logger.error(f"Error loading instruments: {str(e)}")
            return None
    
    def _filter_universe(self):
        """Filter instruments for liquid single-stock options"""
        if self.instruments_df is None:
            return None
        
        # Filter for OPTSTK (single-stock options)
        option_stocks = self.instruments_df[
            (self.instruments_df['instrument_type'] == 'OPTSTK') &
            (self.instruments_df['exchange'] == 'NSE_FO')
        ].copy()
        
        # Get unique underlyings
        underlyings = option_stocks['tradingsymbol'].str.extract(r'([A-Z]+)')[0].unique()
        
        # Filter for liquid underlyings (top 60-90 by volume/options count)
        underlying_counts = option_stocks['tradingsymbol'].str.extract(r'([A-Z]+)')[0].value_counts()
        liquid_underlyings = underlying_counts.head(75).index.tolist()  # Top 75
        
        # Filter for liquid underlyings only
        liquid_options = option_stocks[
            option_stocks['tradingsymbol'].str.extract(r'([A-Z]+)')[0].isin(liquid_underlyings)
        ].copy()
        
        # Get latest expiries for each underlying
        liquid_options['expiry'] = pd.to_datetime(liquid_options['expiry'])
        latest_expiries = liquid_options.groupby('tradingsymbol').str.extract(r'([A-Z]+)')[0].apply(
            lambda x: liquid_options[liquid_options['tradingsymbol'].str.extract(r'([A-Z]+)')[0] == x]['expiry'].max()
        )
        
        # Filter for latest expiries
        self.option_universe = liquid_options[
            liquid_options.apply(
                lambda row: row['expiry'] == latest_expiries.get(row['tradingsymbol'].str.extract(r'([A-Z]+)')[0].iloc[0]),
                axis=1
            )
        ].copy()
        
        # Add DTE
        self.option_universe['dte'] = self.option_universe['expiry'].apply(
            lambda x: (x.date() - datetime.now().date()).days
        )
        
        # Filter for reasonable DTE (15-45 days)
        self.option_universe = self.option_universe[
            (self.option_universe['dte'] >= 15) & 
            (self.option_universe['dte'] <= 45)
        ].copy()
        
        logger.info(f"Filtered universe: {len(self.option_universe)} options across {len(liquid_underlyings)} underlyings")
        return self.option_universe
    
    def get_underlying_list(self):
        """Get list of liquid underlyings"""
        if self.option_universe is None:
            return []
        
        underlyings = self.option_universe['tradingsymbol'].str.extract(r'([A-Z]+)')[0].unique()
        return sorted(underlyings.tolist())

print("✅ Universe Builder defined")

In [ ]:
# WebSocket Manager for live market data
class WebSocketManager:
    def __init__(self, api_client):
        self.api_client = api_client
        self.ws_url = None
        self.websocket = None
        self.subscribed_instruments = set()
        self.live_data = {}
        self.is_connected = False
    
    async def initialize(self):
        """Initialize WebSocket connection"""
        try:
            # Get authorization URL
            auth_response = await self.api_client.get_authorize_url()
            if auth_response.get("status") == "success":
                self.ws_url = auth_response.get("data", {}).get("authorize_redirect_uri")
                logger.info(f"WebSocket URL obtained: {self.ws_url[:50]}...")
                return True
            else:
                logger.error(f"Failed to get WebSocket URL: {auth_response}")
                return False
        except Exception as e:
            logger.error(f"Error initializing WebSocket: {str(e)}")
            return False
    
    async def connect(self):
        """Connect to WebSocket (placeholder - requires actual WebSocket implementation)"""
        # Note: This is a placeholder. Actual WebSocket implementation would require
        # additional libraries like websockets or asyncio websocket client
        logger.info("WebSocket connection placeholder - implement with actual WebSocket client")
        self.is_connected = True
        return True
    
    async def subscribe_instruments(self, instrument_keys: List[str]):
        """Subscribe to live data for instruments"""
        if not self.is_connected:
            logger.warning("WebSocket not connected")
            return False
        
        # Add to subscribed list
        self.subscribed_instruments.update(instrument_keys)
        logger.info(f"Subscribed to {len(instrument_keys)} instruments")
        return True
    
    def get_live_data(self, instrument_key: str) -> Dict:
        """Get live data for an instrument"""
        return self.live_data.get(instrument_key, {})
    
    async def disconnect(self):
        """Disconnect WebSocket"""
        if self.websocket:
            await self.websocket.close()
        self.is_connected = False
        logger.info("WebSocket disconnected")

print("✅ WebSocket Manager defined")

In [ ]:
# Greeks Data Manager
class GreeksDataManager:
    def __init__(self, api_client):
        self.api_client = api_client
        self.greeks_cache = {}
        self.last_update = None
        self.update_interval = 15  # seconds
    
    async def update_greeks(self, instrument_keys: List[str]) -> Dict[str, OptionData]:
        """Update Greeks data for instruments"""
        current_time = time.time()
        
        # Check if update is needed
        if (self.last_update and 
            current_time - self.last_update < self.update_interval):
            return self.greeks_cache
        
        try:
            # Batch instruments (max 50 per request)
            batches = batch_instruments(instrument_keys, 50)
            all_greeks = {}
            
            for batch in batches:
                response = await self.api_client.get_greeks(batch)
                
                if response.get("status") == "success":
                    greeks_data = response.get("data", {})
                    
                    for key, data in greeks_data.items():
                        try:
                            option_data = OptionData(
                                symbol=data.get("tradingsymbol", ""),
                                strike=float(data.get("strike_price", 0)),
                                expiry=data.get("expiry", ""),
                                option_type=data.get("option_type", ""),
                                ltp=float(data.get("ltp", 0)),
                                iv=float(data.get("implied_volatility", 0)),
                                delta=float(data.get("delta", 0)),
                                gamma=float(data.get("gamma", 0)),
                                theta=float(data.get("theta", 0)),
                                vega=float(data.get("vega", 0)),
                                underlying=data.get("tradingsymbol", "").split()[0] if data.get("tradingsymbol") else "",
                                dte=calculate_dte(data.get("expiry", ""))
                            )
                            all_greeks[key] = option_data
                        except (ValueError, KeyError) as e:
                            logger.warning(f"Error parsing Greeks data for {key}: {str(e)}")
                            continue
                else:
                    logger.warning(f"Failed to get Greeks for batch: {response}")
            
            self.greeks_cache = all_greeks
            self.last_update = current_time
            logger.info(f"Updated Greeks for {len(all_greeks)} instruments")
            
            return all_greeks
        
        except Exception as e:
            logger.error(f"Error updating Greeks: {str(e)}")
            return self.greeks_cache
    
    def get_greeks(self, instrument_key: str) -> Optional[OptionData]:
        """Get Greeks data for a specific instrument"""
        return self.greeks_cache.get(instrument_key)

print("✅ Greeks Data Manager defined")

In [ ]:
# Positions Data Manager
class PositionsDataManager:
    def __init__(self, api_client):
        self.api_client = api_client
        self.positions_cache = {}
        self.last_update = None
        self.update_interval = 10  # seconds
    
    async def update_positions(self) -> Dict[str, PositionData]:
        """Update positions data"""
        current_time = time.time()
        
        # Check if update is needed
        if (self.last_update and 
            current_time - self.last_update < self.update_interval):
            return self.positions_cache
        
        try:
            response = await self.api_client.get_positions()
            
            if response.get("status") == "success":
                positions_data = response.get("data", [])
                all_positions = {}
                
                for pos in positions_data:
                    try:
                        position = PositionData(
                            symbol=pos.get("tradingsymbol", ""),
                            quantity=int(pos.get("quantity", 0)),
                            average_price=float(pos.get("average_price", 0)),
                            ltp=float(pos.get("ltp", 0)),
                            pnl=float(pos.get("pnl", 0)),
                            pnl_percentage=float(pos.get("pnl_percentage", 0)),
                            day_pnl=float(pos.get("day_pnl", 0)),
                            day_pnl_percentage=float(pos.get("day_pnl_percentage", 0))
                        )
                        all_positions[pos.get("instrument_key", "")] = position
                    except (ValueError, KeyError) as e:
                        logger.warning(f"Error parsing position data: {str(e)}")
                        continue
                
                self.positions_cache = all_positions
                self.last_update = current_time
                logger.info(f"Updated positions for {len(all_positions)} instruments")
                
                return all_positions
            else:
                logger.warning(f"Failed to get positions: {response}")
                return self.positions_cache
        
        except Exception as e:
            logger.error(f"Error updating positions: {str(e)}")
            return self.positions_cache
    
    def get_positions(self) -> Dict[str, PositionData]:
        """Get current positions"""
        return self.positions_cache
    
    def get_portfolio_summary(self) -> Dict[str, float]:
        """Get portfolio summary"""
        total_pnl = sum(pos.pnl for pos in self.positions_cache.values())
        total_day_pnl = sum(pos.day_pnl for pos in self.positions_cache.values())
        
        return {
            "total_pnl": total_pnl,
            "total_day_pnl": total_day_pnl,
            "position_count": len(self.positions_cache)
        }

print("✅ Positions Data Manager defined")

## 5. Calculations - IVR/IVP, RV/EWMA, VRP, EM, Slope, Skew, ATR

In [ ]:
# Volatility Analytics Engine
class VolatilityAnalytics:
    def __init__(self):
        self.iv_history = {}  # Store 52-week IV history per underlying
        self.price_history = {}  # Store price history for RV calculation
        self.vix_data = None
    
    def calculate_iv_metrics(self, underlying: str, current_iv: float) -> Tuple[float, float]:
        """Calculate IV Rank and IV Percentile for an underlying"""
        if underlying not in self.iv_history:
            return 0.0, 0.0
        
        iv_history = self.iv_history[underlying]
        return calculate_iv_rank(current_iv, iv_history)
    
    def calculate_realized_volatility(self, underlying: str, returns: pd.Series) -> float:
        """Calculate realized volatility using EWMA"""
        return calculate_realized_volatility(returns)
    
    def calculate_vrp(self, iv: float, rv: float) -> float:
        """Calculate Volatility Risk Premium"""
        return iv**2 - rv**2
    
    def calculate_term_slope(self, short_iv: float, long_iv: float) -> float:
        """Calculate term structure slope"""
        return long_iv - short_iv
    
    def calculate_skew(self, call_iv: float, put_iv: float) -> float:
        """Calculate 25-delta risk reversal (skew)"""
        return call_iv - put_iv
    
    def calculate_atr_metrics(self, high: pd.Series, low: pd.Series, close: pd.Series) -> float:
        """Calculate ATR and compare with expected move"""
        return calculate_atr(high, low, close)
    
    def get_vix_regime(self, vix_value: float) -> str:
        """Determine VIX regime based on percentile"""
        if vix_value is None:
            return "UNKNOWN"
        
        # Simple regime classification (would need historical VIX data for proper percentiles)
        if vix_value < 15:
            return "LOW"
        elif vix_value < 25:
            return "NORMAL"
        else:
            return "HIGH"
    
    def determine_bias(self, iv_rank: float, vrp: float, vix_regime: str) -> str:
        """Determine trading bias based on volatility metrics"""
        # GREEN: Long premium bias
        if iv_rank < 0.25 and vrp > 0 and vix_regime == "LOW":
            return "GREEN"
        
        # RED: Short premium bias
        elif iv_rank > 0.75 and vrp < 0 and vix_regime == "HIGH":
            return "RED"
        
        # SKY-BLUE: Neutral/hedge bias
        else:
            return "SKY-BLUE"

print("✅ Volatility Analytics Engine defined")

In [ ]:
# Portfolio Analytics Engine
class PortfolioAnalytics:
    def __init__(self):
        self.positions = {}
        self.greeks = {}
        self.underlying_prices = {}
    
    def calculate_net_greeks(self, positions: Dict[str, PositionData], greeks: Dict[str, OptionData]) -> Dict[str, float]:
        """Calculate net Greeks for portfolio"""
        net_delta = 0.0
        net_gamma = 0.0
        net_theta = 0.0
        net_vega = 0.0
        
        for pos_key, position in positions.items():
            if pos_key in greeks:
                option_data = greeks[pos_key]
                quantity = position.quantity
                
                net_delta += option_data.delta * quantity
                net_gamma += option_data.gamma * quantity
                net_theta += option_data.theta * quantity
                net_vega += option_data.vega * quantity
        
        return {
            "delta": net_delta,
            "gamma": net_gamma,
            "theta": net_theta,
            "vega": net_vega
        }
    
    def calculate_stress_pnl(self, positions: Dict[str, PositionData], greeks: Dict[str, OptionData], 
                           spot_change: float, iv_change: float, time_decay: float) -> float:
        """Calculate P&L under stress scenarios"""
        stress_pnl = 0.0
        
        for pos_key, position in positions.items():
            if pos_key in greeks:
                option_data = greeks[pos_key]
                quantity = position.quantity
                
                # Delta P&L
                delta_pnl = option_data.delta * spot_change * quantity
                
                # Gamma P&L (approximation)
                gamma_pnl = 0.5 * option_data.gamma * (spot_change ** 2) * quantity
                
                # Theta P&L
                theta_pnl = option_data.theta * time_decay * quantity
                
                # Vega P&L
                vega_pnl = option_data.vega * iv_change * quantity
                
                stress_pnl += delta_pnl + gamma_pnl + theta_pnl + vega_pnl
        
        return stress_pnl
    
    def calculate_hedge_size(self, net_delta: float, underlying_price: float, lot_size: int = 25) -> int:
        """Calculate delta hedge size in lots"""
        if underlying_price == 0:
            return 0
        
        hedge_value = abs(net_delta) * underlying_price
        hedge_lots = int(hedge_value / (underlying_price * lot_size))
        
        return hedge_lots if net_delta < 0 else -hedge_lots

print("✅ Portfolio Analytics Engine defined")

## 6. IV-RV Context Dashboard UI

In [ ]:
# IV-RV Context Dashboard
class IVRVDashboard:
    def __init__(self, universe_builder, volatility_analytics):
        self.universe_builder = universe_builder
        self.volatility_analytics = volatility_analytics
        self.selected_underlying = None
        self.metrics_data = {}
    
    def create_dashboard(self):
        """Create the IV-RV Context Dashboard"""
        # Underlying selector
        underlying_options = self.universe_builder.get_underlying_list()
        self.underlying_selector = widgets.Dropdown(
            options=underlying_options,
            description='Underlying:',
            style={'description_width': 'initial'}
        )
        
        # Refresh button
        refresh_button = widgets.Button(
            description='Refresh Data',
            button_style='info',
            icon='refresh'
        )
        
        # Metrics display
        self.metrics_output = widgets.Output()
        
        # Charts
        self.charts_output = widgets.Output()
        
        # Layout
        dashboard = widgets.VBox([
            widgets.HBox([self.underlying_selector, refresh_button]),
            self.metrics_output,
            self.charts_output
        ])
        
        # Event handlers
        self.underlying_selector.observe(self.on_underlying_change, names='value')
        refresh_button.on_click(self.on_refresh_click)
        
        return dashboard
    
    def on_underlying_change(self, change):
        """Handle underlying selection change"""
        if change['new']:
            self.selected_underlying = change['new']
            self.update_dashboard()
    
    def on_refresh_click(self, button):
        """Handle refresh button click"""
        self.update_dashboard()
    
    def update_dashboard(self):
        """Update dashboard with current data"""
        if not self.selected_underlying:
            return
        
        with self.metrics_output:
            clear_output(wait=True)
            self.display_metrics()
        
        with self.charts_output:
            clear_output(wait=True)
            self.display_charts()
    
    def display_metrics(self):
        """Display volatility metrics"""
        if self.selected_underlying not in self.metrics_data:
            print(f"No data available for {self.selected_underlying}")
            return
        
        metrics = self.metrics_data[self.selected_underlying]
        
        # Create metrics display
        metrics_html = f"""
        <div style='display: flex; flex-wrap: wrap; gap: 10px; margin: 10px 0;'>
            <div style='background: #f0f0f0; padding: 10px; border-radius: 5px; min-width: 120px;'>
                <strong>IV</strong><br>{metrics.get('iv', 0):.2%}
            </div>
            <div style='background: #e6f3ff; padding: 10px; border-radius: 5px; min-width: 120px;'>
                <strong>IVR</strong><br>{metrics.get('iv_rank', 0):.1%}
            </div>
            <div style='background: #e6f3ff; padding: 10px; border-radius: 5px; min-width: 120px;'>
                <strong>IVP</strong><br>{metrics.get('iv_percentile', 0):.1%}
            </div>
            <div style='background: #f0f8ff; padding: 10px; border-radius: 5px; min-width: 120px;'>
                <strong>RV</strong><br>{metrics.get('realized_vol', 0):.2%}
            </div>
            <div style='background: #fff0f0; padding: 10px; border-radius: 5px; min-width: 120px;'>
                <strong>VRP</strong><br>{metrics.get('vrp', 0):.4f}
            </div>
            <div style='background: #f0fff0; padding: 10px; border-radius: 5px; min-width: 120px;'>
                <strong>EM(30D)</strong><br>{metrics.get('expected_move', 0):.2f}
            </div>
            <div style='background: #fff8f0; padding: 10px; border-radius: 5px; min-width: 120px;'>
                <strong>Term Slope</strong><br>{metrics.get('term_slope', 0):.2%}
            </div>
            <div style='background: #f8f0ff; padding: 10px; border-radius: 5px; min-width: 120px;'>
                <strong>Skew</strong><br>{metrics.get('skew', 0):.2%}
            </div>
            <div style='background: #f0f0f0; padding: 10px; border-radius: 5px; min-width: 120px;'>
                <strong>ATR vs EM</strong><br>{metrics.get('atr_vs_em', 0):.2f}
            </div>
            <div style='background: #e6ffe6; padding: 10px; border-radius: 5px; min-width: 120px;'>
                <strong>VIX Regime</strong><br>{metrics.get('vix_regime', 'UNKNOWN')}
            </div>
            <div style='background: #ffe6e6; padding: 10px; border-radius: 5px; min-width: 120px;'>
                <strong>Bias</strong><br><span style='color: {self.get_bias_color(metrics.get('bias', 'SKY-BLUE'))}'>{metrics.get('bias', 'SKY-BLUE')}</span>
            </div>
        </div>
        """
        
        display(widgets.HTML(metrics_html))
    
    def get_bias_color(self, bias: str) -> str:
        """Get color for bias display"""
        colors = {
            'GREEN': 'green',
            'SKY-BLUE': 'skyblue',
            'RED': 'red'
        }
        return colors.get(bias, 'gray')
    
    def display_charts(self):
        """Display volatility charts"""
        if self.selected_underlying not in self.metrics_data:
            print(f"No chart data available for {self.selected_underlying}")
            return
        
        # Placeholder for charts
        print(f"Charts for {self.selected_underlying} would be displayed here")
        print("(Price with EM bands, IV term structure, Skew chart, ATR vs EM)")

print("✅ IV-RV Context Dashboard defined")

## 7. Trade-Position Monitor UI

In [ ]:
# Trade-Position Monitor
class PositionMonitor:
    def __init__(self, positions_manager, portfolio_analytics):
        self.positions_manager = positions_manager
        self.portfolio_analytics = portfolio_analytics
        self.stress_scenarios = {
            'spot_1pct': 0.01,
            'spot_em_half': 0.0,  # Will be calculated
            'iv_5pts': 0.05,
            'time_1day': 1.0
        }
    
    def create_monitor(self):
        """Create the Position Monitor UI"""
        # P&L Summary
        self.pnl_output = widgets.Output()
        
        # Greeks Summary
        self.greeks_output = widgets.Output()
        
        # Stress Test Grid
        self.stress_output = widgets.Output()
        
        # Action Prompts
        self.actions_output = widgets.Output()
        
        # Refresh button
        refresh_button = widgets.Button(
            description='Refresh Positions',
            button_style='success',
            icon='refresh'
        )
        refresh_button.on_click(self.on_refresh_click)
        
        # Layout
        monitor = widgets.VBox([
            refresh_button,
            self.pnl_output,
            self.greeks_output,
            self.stress_output,
            self.actions_output
        ])
        
        return monitor
    
    def on_refresh_click(self, button):
        """Handle refresh button click"""
        self.update_monitor()
    
    def update_monitor(self):
        """Update monitor with current data"""
        with self.pnl_output:
            clear_output(wait=True)
            self.display_pnl_summary()
        
        with self.greeks_output:
            clear_output(wait=True)
            self.display_greeks_summary()
        
        with self.stress_output:
            clear_output(wait=True)
            self.display_stress_grid()
        
        with self.actions_output:
            clear_output(wait=True)
            self.display_action_prompts()
    
    def display_pnl_summary(self):
        """Display P&L summary"""
        summary = self.positions_manager.get_portfolio_summary()
        
        pnl_html = f"""
        <div style='display: flex; gap: 20px; margin: 10px 0;'>
            <div style='background: #e6f3ff; padding: 15px; border-radius: 8px; min-width: 150px;'>
                <h4>Total P&L</h4>
                <p style='font-size: 24px; font-weight: bold; color: {'green' if summary['total_pnl'] >= 0 else 'red'};'>
                    ₹{summary['total_pnl']:,.2f}
                </p>
            </div>
            <div style='background: #e6ffe6; padding: 15px; border-radius: 8px; min-width: 150px;'>
                <h4>Day P&L</h4>
                <p style='font-size: 24px; font-weight: bold; color: {'green' if summary['total_day_pnl'] >= 0 else 'red'};'>
                    ₹{summary['total_day_pnl']:,.2f}
                </p>
            </div>
            <div style='background: #fff0e6; padding: 15px; border-radius: 8px; min-width: 150px;'>
                <h4>Positions</h4>
                <p style='font-size: 24px; font-weight: bold;'>
                    {summary['position_count']}
                </p>
            </div>
        </div>
        """
        
        display(widgets.HTML(pnl_html))
    
    def display_greeks_summary(self):
        """Display Greeks summary"""
        positions = self.positions_manager.get_positions()
        # Note: Would need Greeks data from GreeksDataManager
        
        greeks_html = """
        <div style='background: #f8f9fa; padding: 15px; border-radius: 8px; margin: 10px 0;'>
            <h4>Net Greeks</h4>
            <p>Greeks data would be displayed here when available</p>
        </div>
        """
        
        display(widgets.HTML(greeks_html))
    
    def display_stress_grid(self):
        """Display stress test grid"""
        stress_html = """
        <div style='background: #f8f9fa; padding: 15px; border-radius: 8px; margin: 10px 0;'>
            <h4>Stress Test Grid</h4>
            <p>Stress test results would be displayed here</p>
        </div>
        """
        
        display(widgets.HTML(stress_html))
    
    def display_action_prompts(self):
        """Display action prompts"""
        actions_html = """
        <div style='background: #f8f9fa; padding: 15px; border-radius: 8px; margin: 10px 0;'>
            <h4>Action Prompts</h4>
            <p>Delta hedge size, roll suggestions, and harvest plans would be displayed here</p>
        </div>
        """
        
        display(widgets.HTML(actions_html))

print("✅ Position Monitor defined")

## 8. Main Dashboard - Integration and Control

In [ ]:
# Main Dashboard Controller
class MainDashboard:
    def __init__(self, config):
        self.config = config
        self.api_client = None
        self.universe_builder = None
        self.volatility_analytics = None
        self.portfolio_analytics = None
        self.iv_rv_dashboard = None
        self.position_monitor = None
        self.is_initialized = False
    
    async def initialize(self):
        """Initialize all components"""
        try:
            # Initialize API client
            self.api_client = UpstoxAPIClient(self.config)
            
            # Initialize components
            self.universe_builder = UniverseBuilder(self.api_client)
            self.volatility_analytics = VolatilityAnalytics()
            self.portfolio_analytics = PortfolioAnalytics()
            
            # Load universe
            await self.universe_builder.load_instruments()
            
            # Initialize UI components
            self.iv_rv_dashboard = IVRVDashboard(self.universe_builder, self.volatility_analytics)
            self.position_monitor = PositionMonitor(None, self.portfolio_analytics)  # Would need positions manager
            
            self.is_initialized = True
            logger.info("Dashboard initialized successfully")
            
        except Exception as e:
            logger.error(f"Failed to initialize dashboard: {str(e)}")
            self.is_initialized = False
    
    def create_main_dashboard(self):
        """Create the main dashboard layout"""
        if not self.is_initialized:
            return widgets.HTML("<p>Dashboard not initialized. Please run the initialization cell first.</p>")
        
        # Create left pane (IV-RV Context)
        left_pane = widgets.VBox([
            widgets.HTML("<h3>IV-RV Context Dashboard</h3>"),
            self.iv_rv_dashboard.create_dashboard()
        ])
        
        # Create right pane (Position Monitor)
        right_pane = widgets.VBox([
            widgets.HTML("<h3>Trade-Position Monitor</h3>"),
            self.position_monitor.create_monitor()
        ])
        
        # Main layout
        main_dashboard = widgets.HBox([
            widgets.VBox([left_pane], layout=widgets.Layout(width='50%')),
            widgets.VBox([right_pane], layout=widgets.Layout(width='50%'))
        ])
        
        return main_dashboard

print("✅ Main Dashboard Controller defined")

## 9. Self-Tests and Validation

In [ ]:
# Self-Tests and Validation
class DashboardTests:
    def __init__(self, dashboard):
        self.dashboard = dashboard
        self.test_results = {}
    
    async def run_all_tests(self):
        """Run all acceptance tests"""
        print("🧪 Running Dashboard Tests...")
        
        # Test 1: Universe loading
        await self.test_universe_loading()
        
        # Test 2: API connectivity
        await self.test_api_connectivity()
        
        # Test 3: Calculations
        self.test_calculations()
        
        # Test 4: UI components
        self.test_ui_components()
        
        # Display results
        self.display_test_results()
    
    async def test_universe_loading(self):
        """Test universe loading"""
        try:
            if self.dashboard.universe_builder:
                universe = await self.dashboard.universe_builder.load_instruments()
                if universe is not None and len(universe) > 0:
                    self.test_results['universe_loading'] = {
                        'status': 'PASS',
                        'message': f'Loaded {len(universe)} instruments'
                    }
                else:
                    self.test_results['universe_loading'] = {
                        'status': 'FAIL',
                        'message': 'No instruments loaded'
                    }
            else:
                self.test_results['universe_loading'] = {
                    'status': 'SKIP',
                    'message': 'Universe builder not initialized'
                }
        except Exception as e:
            self.test_results['universe_loading'] = {
                'status': 'FAIL',
                'message': f'Error: {str(e)}'
            }
    
    async def test_api_connectivity(self):
        """Test API connectivity"""
        try:
            if self.dashboard.api_client:
                async with self.dashboard.api_client as client:
                    # Test a simple API call
                    response = await client.get_positions()
                    if response:
                        self.test_results['api_connectivity'] = {
                            'status': 'PASS',
                            'message': 'API connection successful'
                        }
                    else:
                        self.test_results['api_connectivity'] = {
                            'status': 'FAIL',
                            'message': 'API response empty'
                        }
            else:
                self.test_results['api_connectivity'] = {
                    'status': 'SKIP',
                    'message': 'API client not initialized'
                }
        except Exception as e:
            self.test_results['api_connectivity'] = {
                'status': 'FAIL',
                'message': f'API error: {str(e)}'
            }
    
    def test_calculations(self):
        """Test calculation functions"""
        try:
            # Test IV rank calculation
            iv_history = [0.15, 0.20, 0.25, 0.30, 0.35]
            iv_rank, iv_percentile = calculate_iv_rank(0.25, iv_history)
            
            if 0 <= iv_rank <= 1 and 0 <= iv_percentile <= 1:
                self.test_results['calculations'] = {
                    'status': 'PASS',
                    'message': 'Calculation functions working correctly'
                }
            else:
                self.test_results['calculations'] = {
                    'status': 'FAIL',
                    'message': 'Calculation results out of range'
                }
        except Exception as e:
            self.test_results['calculations'] = {
                'status': 'FAIL',
                'message': f'Calculation error: {str(e)}'
            }
    
    def test_ui_components(self):
        """Test UI components"""
        try:
            if self.dashboard.iv_rv_dashboard and self.dashboard.position_monitor:
                self.test_results['ui_components'] = {
                    'status': 'PASS',
                    'message': 'UI components initialized successfully'
                }
            else:
                self.test_results['ui_components'] = {
                    'status': 'FAIL',
                    'message': 'UI components not initialized'
                }
        except Exception as e:
            self.test_results['ui_components'] = {
                'status': 'FAIL',
                'message': f'UI error: {str(e)}'
            }
    
    def display_test_results(self):
        """Display test results"""
        print("\n📊 Test Results:")
        print("=" * 50)
        
        for test_name, result in self.test_results.items():
            status_emoji = {
                'PASS': '✅',
                'FAIL': '❌',
                'SKIP': '⏭️'
            }
            
            print(f"{status_emoji.get(result['status'], '❓')} {test_name.replace('_', ' ').title()}: {result['message']}")
        
        print("=" * 50)
        
        # Overall status
        passed = sum(1 for r in self.test_results.values() if r['status'] == 'PASS')
        total = len(self.test_results)
        
        print(f"\n🎯 Overall: {passed}/{total} tests passed")
        
        if passed == total:
            print("🎉 All tests passed! Dashboard is ready to use.")
        else:
            print("⚠️  Some tests failed. Please check the configuration and try again.")

print("✅ Dashboard Tests defined")

## 10. Usage Instructions and Final Setup

In [ ]:
# Initialize and run the dashboard
async def initialize_dashboard():
    """Initialize the complete dashboard"""
    print("🚀 Initializing Upstox Trade Monitor Dashboard...")
    
    # Create main dashboard
    main_dashboard = MainDashboard(config)
    
    # Initialize components
    await main_dashboard.initialize()
    
    # Run tests
    tests = DashboardTests(main_dashboard)
    await tests.run_all_tests()
    
    # Create and display dashboard
    dashboard_ui = main_dashboard.create_main_dashboard()
    
    print("\n📊 Dashboard Ready!")
    print("\nInstructions:")
    print("1. Use the left pane to select underlyings and view IV-RV context")
    print("2. Use the right pane to monitor positions and P&L")
    print("3. Click refresh buttons to update data")
    print("4. Check the action prompts for trading suggestions")
    
    return dashboard_ui

# Run the dashboard
dashboard = await initialize_dashboard()
display(dashboard)